In [ ]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
import torch.nn as nn
import numpy as np
import pathlib

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 디바이스: {DEVICE}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

# ===== LSTM Autoencoder 모델 =====

class Encoder(nn.Module):
    def __init__(self, input_dim=8, hidden_dim=64, latent_dim=32, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2
        )
        self.fc = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x):
        # x: (batch, seq_len, input_dim)
        out, (h, c) = self.lstm(x)
        # 마지막 타임스텝의 hidden state
        latent = self.fc(h[-1])  # (batch, latent_dim)
        return latent

class Decoder(nn.Module):
    def __init__(self, latent_dim=32, hidden_dim=64, output_dim=8, seq_len=60, num_layers=2):
        super().__init__()
        self.seq_len = seq_len
        self.fc = nn.Linear(latent_dim, hidden_dim)
        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2
        )
        self.out = nn.Linear(hidden_dim, output_dim)

    def forward(self, latent):
        # latent: (batch, latent_dim)
        x = self.fc(latent)                        # (batch, hidden_dim)
        x = x.unsqueeze(1).repeat(1, self.seq_len, 1)  # (batch, seq_len, hidden_dim)
        out, _ = self.lstm(x)                      # (batch, seq_len, hidden_dim)
        return self.out(out)                        # (batch, seq_len, output_dim)

class LSTMAutoencoder(nn.Module):
    def __init__(self, input_dim=8, hidden_dim=64, latent_dim=32, seq_len=60, num_layers=2):
        super().__init__()
        self.encoder = Encoder(input_dim, hidden_dim, latent_dim, num_layers)
        self.decoder = Decoder(latent_dim, hidden_dim, input_dim, seq_len, num_layers)

    def forward(self, x):
        latent = self.encoder(x)
        recon = self.decoder(latent)
        return recon

    def encode(self, x):
        return self.encoder(x)

# 모델 생성 및 테스트
model = LSTMAutoencoder(
    input_dim=8,
    hidden_dim=64,
    latent_dim=32,
    seq_len=60,
    num_layers=2
).to(DEVICE)

# 파라미터 수 확인
total_params = sum(p.numel() for p in model.parameters())
print(f"\n모델 파라미터 수: {total_params:,}개")

# Forward pass 테스트
dummy = torch.randn(4, 60, 8).to(DEVICE)  # batch=4, seq=60, dim=8
output = model(dummy)
print(f"\nForward pass 테스트:")
print(f"  입력 shape: {dummy.shape}")
print(f"  출력 shape: {output.shape}")
print(f"  ✅ 정상 동작 확인!")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pathlib
from scipy.interpolate import interp1d

NORMALIZED_DIR = pathlib.Path.home() / "taekwondo-scorer/processed/normalized"
SEQ_LEN = 60

class TaegeukDataset(Dataset):
    def __init__(self, action_name, mode="train", train_ratio=0.8):
        self.samples = []

        # 해당 동작 파일 전부 로딩
        all_files = list((NORMALIZED_DIR / action_name).glob("*/*.npy"))
        print(f"  {action_name}: {len(all_files)}개 파일 발견")

        # train/val 분할
        split = int(len(all_files) * train_ratio)
        if mode == "train":
            files = all_files[:split]
        else:
            files = all_files[split:]

        # 각도값 범위 파악용 (정규화에 사용)
        for f in files:
            try:
                seq = np.load(f)  # (T, 8)
                if len(seq) == 0:
                    continue
                # 60프레임으로 리샘플링
                if len(seq) == 1:
                    seq = np.tile(seq, (SEQ_LEN, 1))
                else:
                    interp = interp1d(np.linspace(0, 1, len(seq)), seq, axis=0)
                    seq = interp(np.linspace(0, 1, SEQ_LEN))

                # 각도를 0~1로 정규화 (0~180도 범위)
                seq = seq / 180.0

                self.samples.append(seq.astype(np.float32))
            except:
                continue

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return torch.tensor(self.samples[idx])  # (60, 8)

# 전체 동작 데이터셋 구성
ACTION_NAMES = [
    "기본준비", "앞굽이하고 아래막고 지르기", "앞굽이하고 아래막기",
    "앞굽이하고 지르기", "앞서고 아래막기", "앞서고 안막기",
    "앞서고 얼굴막기", "앞서고 지르기", "앞차고 앞서고 지르기"
]

print("데이터셋 구성 중...\n")
train_loaders = {}
val_loaders = {}

for action in ACTION_NAMES:
    train_ds = TaegeukDataset(action, mode="train")
    val_ds   = TaegeukDataset(action, mode="val")

    train_loaders[action] = DataLoader(
        train_ds, batch_size=256, shuffle=True, num_workers=0
    )
    val_loaders[action] = DataLoader(
        val_ds, batch_size=256, shuffle=False, num_workers=0
    )

    print(f"  ✅ {action}: train {len(train_ds)}개 / val {len(val_ds)}개")

# 배치 하나 꺼내서 shape 확인
sample_action = "앞서고 지르기"
batch = next(iter(train_loaders[sample_action]))
print(f"\n배치 shape 확인: {batch.shape}")  # (256, 60, 8) 이어야 함
print("✅ 데이터셋 구성 완료!")

In [ ]:
import torch.optim as optim
import time
import matplotlib.pyplot as plt

MODEL_DIR = pathlib.Path.home() / "taekwondo-scorer/models"
MODEL_DIR.mkdir(exist_ok=True)

criterion = nn.MSELoss()

def train_one_action(action_name, epochs=50, patience=10):
    print(f"\n{'='*50}")
    print(f"학습 시작: {action_name}")
    print(f"{'='*50}")

    # 모델 생성
    model = LSTMAutoencoder(
        input_dim=8, hidden_dim=64, latent_dim=32,
        seq_len=60, num_layers=2
    ).to(DEVICE)

    optimizer = optim.Adam(model.parameters(), lr=0.001)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=5, factor=0.5, verbose=True
    )

    train_loader = train_loaders[action_name]
    val_loader   = val_loaders[action_name]

    best_val_loss = float('inf')
    patience_counter = 0
    train_losses, val_losses = [], []

    for epoch in range(epochs):
        # Train
        model.train()
        train_loss = 0
        for batch in train_loader:
            batch = batch.to(DEVICE)
            optimizer.zero_grad()
            output = model(batch)
            loss = criterion(output, batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(DEVICE)
                output = model(batch)
                loss = criterion(output, batch)
                val_loss += loss.item()
        val_loss /= len(val_loader)

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        scheduler.step(val_loss)

        # 10 에폭마다 출력
        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1:3d}/{epochs} | "
                  f"Train: {train_loss:.6f} | Val: {val_loss:.6f}")

        # Best 모델 저장
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            save_path = MODEL_DIR / f"{action_name}_autoencoder.pth"
            torch.save(model.state_dict(), save_path)
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    print(f"  ✅ 완료! Best Val Loss: {best_val_loss:.6f}")
    print(f"  저장: {MODEL_DIR / f'{action_name}_autoencoder.pth'}")
    return train_losses, val_losses

# 전체 9개 동작 순서대로 학습
all_losses = {}
start_time = time.time()

for action in ACTION_NAMES:
    losses = train_one_action(action, epochs=50, patience=10)
    all_losses[action] = losses

total_time = time.time() - start_time
print(f"\n{'='*50}")
print(f"전체 학습 완료! 총 소요 시간: {total_time/60:.1f}분")
print(f"저장된 모델: {MODEL_DIR}")

In [ ]:
import torch
import numpy as np
import pathlib
import json
from scipy.interpolate import interp1d

DEVICE = torch.device("cuda")
MODEL_DIR = pathlib.Path.home() / "taekwondo-scorer/models"
NORMALIZED_DIR = pathlib.Path.home() / "taekwondo-scorer/processed/normalized"

ACTION_NAMES = [
    "기본준비", "앞굽이하고 아래막고 지르기", "앞굽이하고 아래막기",
    "앞굽이하고 지르기", "앞서고 아래막기", "앞서고 안막기",
    "앞서고 얼굴막기", "앞서고 지르기", "앞차고 앞서고 지르기"
]

# 모델 전부 로딩
ae_models = {}
for action in ACTION_NAMES:
    model = LSTMAutoencoder(
        input_dim=8, hidden_dim=64, latent_dim=32,
        seq_len=60, num_layers=2
    ).to(DEVICE)
    model_path = MODEL_DIR / f"{action}_autoencoder.pth"
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.eval()
    ae_models[action] = model
    print(f"  ✅ {action} 모델 로딩 완료")

print(f"\n총 {len(ae_models)}개 모델 로딩 완료!")

# 동작별 정상 복원 오차 분포 계산 (점수 매핑 기준선)
print("\n정상 오차 분포 계산 중...")
error_stats = {}

for action in ACTION_NAMES:
    model = ae_models[action]
    all_files = list((NORMALIZED_DIR / action).glob("*/*.npy"))[:300]
    errors = []

    with torch.no_grad():
        for f in all_files:
            try:
                seq = np.load(f)
                if len(seq) == 1:
                    seq = np.tile(seq, (60, 1))
                else:
                    interp = interp1d(np.linspace(0, 1, len(seq)), seq, axis=0)
                    seq = interp(np.linspace(0, 1, 60))
                seq = (seq / 180.0).astype(np.float32)

                tensor = torch.tensor(seq).unsqueeze(0).to(DEVICE)
                output = model(tensor)
                error = torch.nn.functional.mse_loss(output, tensor).item()
                errors.append(error)
            except:
                continue

    errors = np.array(errors)
    error_stats[action] = {
        "min": float(errors.min()),
        "p10": float(np.percentile(errors, 10)),
        "p25": float(np.percentile(errors, 25)),
        "p50": float(np.percentile(errors, 50)),
        "p75": float(np.percentile(errors, 75)),
        "p90": float(np.percentile(errors, 90)),
        "max": float(errors.max()),
    }
    print(f"  {action}: min={errors.min():.5f}, p50={np.percentile(errors,50):.5f}, max={errors.max():.5f}")

# 저장
STATS_PATH = pathlib.Path.home() / "taekwondo-scorer/error_stats.json"
with open(STATS_PATH, "w") as f:
    json.dump(error_stats, f, indent=2)
print(f"\n오차 통계 저장 완료: {STATS_PATH}")

In [ ]:
import json

# error_stats 로딩
STATS_PATH = pathlib.Path.home() / "taekwondo-scorer/error_stats.json"
with open(STATS_PATH, "r") as f:
    error_stats = json.load(f)

JOINT_NAMES = ["왼팔꿈치", "오른팔꿈치", "왼어깨", "오른어깨",
               "왼무릎", "오른무릎", "왼엉덩이", "오른엉덩이"]

def resample_to_tensor(seq, target_len=60):
    if len(seq) == 1:
        seq = np.tile(seq, (target_len, 1))
    else:
        from scipy.interpolate import interp1d
        interp = interp1d(np.linspace(0, 1, len(seq)), seq, axis=0)
        seq = interp(np.linspace(0, 1, target_len))
    seq = (seq / 180.0).astype(np.float32)
    return torch.tensor(seq).unsqueeze(0).to(DEVICE)

def calc_score_from_error(error, stats):
    if error <= stats['min']:
        return 100.0
    elif error <= stats['p10']:
        return 85 + 15 * (1 - (error - stats['min']) / (stats['p10'] - stats['min'] + 1e-8))
    elif error <= stats['p25']:
        return 70 + 15 * (1 - (error - stats['p10']) / (stats['p25'] - stats['p10'] + 1e-8))
    elif error <= stats['p50']:
        return 55 + 15 * (1 - (error - stats['p25']) / (stats['p50'] - stats['p25'] + 1e-8))
    elif error <= stats['p75']:
        return 40 + 15 * (1 - (error - stats['p50']) / (stats['p75'] - stats['p50'] + 1e-8))
    elif error <= stats['p90']:
        return 20 + 20 * (1 - (error - stats['p75']) / (stats['p90'] - stats['p75'] + 1e-8))
    else:
        return max(0, 20 * (1 - (error - stats['p90']) / (stats['max'] - stats['p90'] + 1e-8)))

def score_with_model(seq, action_name):
    """
    seq: (T, 8) 관절 각도 시퀀스
    action_name: 동작 이름
    반환: 점수(0~100), 관절별 오차, 복원 오차
    """
    model = ae_models[action_name]
    stats = error_stats[action_name]

    tensor = resample_to_tensor(seq)

    with torch.no_grad():
        output = model(tensor)

    # 전체 복원 오차
    recon_error = torch.nn.functional.mse_loss(output, tensor).item()

    # 관절별 오차 (마지막 프레임 기준)
    joint_errors = torch.abs(output[0, -1] - tensor[0, -1]).cpu().numpy()
    joint_errors_deg = {name: round(float(e * 180), 2)
                        for name, e in zip(JOINT_NAMES, joint_errors)}

    score = calc_score_from_error(recon_error, stats)
    worst_joint = max(joint_errors_deg, key=joint_errors_deg.get)

    return round(score, 1), joint_errors_deg, round(recon_error, 6), worst_joint

# ===== 테스트 =====
print("=" * 50)
print("모델 기반 채점 테스트")
print("=" * 50)

action_name = "앞서고 지르기"
all_files = list((NORMALIZED_DIR / action_name).glob("*/*.npy"))

print(f"\n{action_name} 샘플 10개 채점:")
scores = []
for f in all_files[:10]:
    seq = np.load(f)
    score, joint_errors, recon_error, worst = score_with_model(seq, action_name)
    scores.append(score)
    print(f"  점수: {score:5.1f}점  복원오차: {recon_error:.6f}  worst: {worst}")

print(f"\n평균 점수: {np.mean(scores):.1f}점")
print(f"최고 점수: {max(scores):.1f}점")
print(f"최저 점수: {min(scores):.1f}점")